# Data Quality Check

In [196]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

import os 
os.getcwd()

'c:\\Users\\glmar\\OneDrive\\Bemacs Bocconi\\TESI\\Thesis\\Priced-by-intelligence'

---
## 1. Option Prices — AAPL

In [197]:
df = pd.read_csv("data/raw/AAPL_options_prices.csv")
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head()

Rows: 1,994,694
Columns: ['secid', 'date', 'exdate', 'cp_flag', 'strike_price', 'best_bid', 'best_offer', 'volume', 'open_interest', 'optionid', 'cfadj', 'ss_flag', 'index_flag', 'issuer', 'exercise_style']


,secid,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,optionid,cfadj,ss_flag,index_flag,issuer,exercise_style
0,101594,2020-08-31,2020-09-04,C,100000,29.40,29.70,150,411,135212375,4,0,0,APPLE INC,A
1,101594,2020-08-31,2020-09-04,C,100630,28.85,29.10,691,81,135212376,4,0,0,APPLE INC,A
2,101594,2020-08-31,2020-09-04,C,101250,28.15,28.45,137,240,135212377,4,0,0,APPLE INC,A
3,101594,2020-08-31,2020-09-04,C,101880,27.50,27.85,75,130,135212378,4,0,0,APPLE INC,A
4,101594,2020-08-31,2020-09-04,C,102500,26.90,27.20,323,734,135212379,4,0,0,APPLE INC,A


In [198]:
missing = df.isna().sum() # column sum
missing = missing[missing > 0] # filter keeping columns with at least a missing value
pct = (missing / len(df) * 100).round(2) # percentage
pd.DataFrame({"missing_count": missing, "missing_pct": pct}).sort_values("missing_pct", ascending=False)

,missing_count,missing_pct


### 'date'

In [199]:
dates = pd.to_datetime(df["date"], errors="coerce")
print(f"Range date: {dates.min().date()} -> {dates.max().date()}")
print(f"Unique days present: {dates.dt.date.nunique()}")

Range date: 2020-08-31 -> 2025-08-29
Unique days present: 1256


Check if dataset has all trading days that should have in the chosen time period

In [200]:
span_years = (dates.max() - dates.min()).days / 365.25 # leap years
expected_approx = int(span_years * 252) # total trading days
print(f"Expected trading days (estimate): ~{expected_approx}")
print(f"Actual days present: {dates.dt.date.nunique()}")
if dates.dt.date.nunique() < expected_approx * 0.85:
    print("[!] WARNING: coverage significantly below expected")

Expected trading days (estimate): ~1258
Actual days present: 1256


### 'exdate'

In [201]:
exdate = pd.to_datetime(df["exdate"], errors="coerce")
date = pd.to_datetime(df["date"], errors="coerce")
days_to_exp = (exdate - date).dt.days

print(f"Day to expiration: min={days_to_exp.min()}, max={days_to_exp.max()}, media={days_to_exp.mean():.0f}")
n_over_year = (days_to_exp > 365).sum()
print(f"Rows with expiration > 365 days: {n_over_year}")
short_expiration = (days_to_exp < 8).sum()
print(f"Rows with expiration <= 7 days: {short_expiration}")

if n_over_year > 0:
    print("[!] WARNING: the query filter did not exclude everything.")

Day to expiration: min=0, max=365, media=102
Rows with expiration > 365 days: 0
Rows with expiration <= 7 days: 202684


### 'cp_flag'

In [202]:
df["cp_flag"].value_counts()

cp_flag
C    997347
P    997347
Name: count, dtype: int64

### 'strike_price'

Visualize range of strikes

In [203]:
print(f"raw strike_price: min={df['strike_price'].min()}, max={df['strike_price'].max()}")
print(f"strike_price /1000: min={df['strike_price'].min()/1000:.2f}, max={df['strike_price'].max()/1000:.2f}")

raw strike_price: min=5000, max=400000
strike_price /1000: min=5.00, max=400.00


In [204]:
strikes = df["strike_price"] / 1000
print(f"5%: {strikes.quantile(0.05):.2f} | 50%: {strikes.quantile(0.5):.2f} | 95%: {strikes.quantile(0.95):.2f}")

5%: 47.50 | 50%: 150.00 | 95%: 295.00


### 'cfadj'

In [205]:
print(df["cfadj"].value_counts(dropna=False))
print()
print(df.groupby("cfadj")["date"].agg(["min", "max", "count"]))

cfadj
1    1715262
4     279432
Name: count, dtype: int64

              min         max    count
cfadj                                 
1      2020-09-02  2025-08-29  1715262
4      2020-08-31  2022-09-16   279432


`cfadj = 4` overlaps in time with `cfadj = 1` from 2020-08-31 to 2022-09-16 (AAPL's 4-for-1 split, ex-date 2020-08-31, per the counts above). If `cfadj = 4` marks pre-split-issued contracts still carrying old-terms strikes, their `strike_price` should look roughly 4x too large relative to both the `cfadj = 1` contracts and the actual AAPL close price over the same window. Check this directly against `AAPL_security_prices.csv`.

In [206]:
strikes = df["strike_price"] / 1000
options_in_overlap = df[(dates >= overlap_start) & (dates <= overlap_end)]
strikes_in_overlap = strikes[(dates >= overlap_start) & (dates <= overlap_end)]

print("\nstrike_price stats by cfadj group, same date window:\n")
print(strikes_in_overlap.groupby(options_in_overlap["cfadj"]).describe())


strike_price stats by cfadj group, same date window:

          count        mean        std    min     25%     50%    75%    max
cfadj                                                                      
1      521480.0  148.881802  52.872103  20.00  115.00  141.00  185.0  300.0
4      279432.0   98.713731  49.150709  18.75   58.75   93.75  127.5  250.0


In [207]:
aapl_prices = pd.read_csv("data/raw/AAPL_security_prices.csv")
aapl_prices["date"] = pd.to_datetime(aapl_prices["date"])

overlap_start, overlap_end = "2020-08-31", "2022-09-16"
dates = pd.to_datetime(df["date"])

close_in_overlap = aapl_prices[
    (aapl_prices["date"] >= overlap_start) & (aapl_prices["date"] <= overlap_end)
]["close"]
print("Actual AAPL close price range in the overlap window:")
print(close_in_overlap.describe())



Actual AAPL close price range in the overlap window:
count    516.000000
mean     143.668314
std       18.799242
min      106.840000
25%      127.805000
50%      144.705000
75%      158.100000
max      182.010000
Name: close, dtype: float64


`strike_price` requires no manual normalization for the AAPL 4-for-1 split (ex-date 2020-08-31) — confirmed via the IvyDB Reference Manual, OCC split-adjustment mechanics (which apply automatically at the exchange level before the data reaches OptionMetrics), and the empirical check (the `cfadj=4` strike distribution already sits on the same scale as `cfadj=1` and matches the actual AAPL close price over the overlap window). No transformation applied to `strike_price` for this column.

### 'best_bid and best_offer'

In [208]:
print(f"negative or zero best_bid: {(df['best_bid'] <= 0).sum()}")
print(f"negative or zero best_offer: {(df['best_offer'] <= 0).sum()}")
n_crossed = (df["best_offer"] < df["best_bid"]).sum()
print(f"Rows with ask < bid (anomalous): {n_crossed}")
if n_crossed > 0:
    print("[!] WARNING: crossed market found in some rows.")

negative or zero best_bid: 196436
negative or zero best_offer: 0
Rows with ask < bid (anomalous): 47
[!] WARNING: crossed market found in some rows.


### 'volume', 'open_interest'

In [209]:
n_total = len(df)
n_zero_vol = (df["volume"] == 0).sum()
n_zero_oi = (df["open_interest"] == 0).sum()
n_both_zero = ((df["volume"] == 0) & (df["open_interest"] == 0)).sum()
n_remaining = n_total - n_both_zero

print(f"Total rows: {n_total:,}")
print(f"volume=0: {n_zero_vol:,} ({n_zero_vol/n_total*100:.1f}%)")
print(f"open_interest=0: {n_zero_oi:,} ({n_zero_oi/n_total*100:.1f}%)")
print(f"Both zero: {n_both_zero:,} ({n_both_zero/n_total*100:.1f}%)")
print(f"Rows remaining after filter: {n_remaining:,} ({n_remaining/n_total*100:.1f}%)")

Total rows: 1,994,694
volume=0: 820,211 (41.1%)
open_interest=0: 332,939 (16.7%)
Both zero: 295,772 (14.8%)
Rows remaining after filter: 1,698,922 (85.2%)


### 'exercise_style'

In [210]:
df["exercise_style"].value_counts()

exercise_style
A    1994694
Name: count, dtype: int64

### 'ss_flag'

In [211]:
df["ss_flag"].value_counts()

ss_flag
0    1994694
Name: count, dtype: int64

### 'index_flag'

In [212]:
df["index_flag"].value_counts()

index_flag
0    1994694
Name: count, dtype: int64

In [213]:
df["_year"] = pd.to_datetime(df["date"]).dt.year
df[df["volume"] > 0].groupby("_year").size()

_year
2020    117555
2021    252596
2022    238638
2023    201401
2024    203910
2025    160383
dtype: int64

---
## 2. Option Prices — SPX

In [214]:
df = pd.read_csv("data/raw/SPX_options_prices.csv")
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head()

Rows: 24,470,893
Columns: ['secid', 'date', 'exdate', 'cp_flag', 'strike_price', 'best_bid', 'best_offer', 'volume', 'open_interest', 'optionid', 'cfadj', 'ss_flag', 'index_flag', 'issuer', 'exercise_style']


,secid,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,optionid,cfadj,ss_flag,index_flag,issuer,exercise_style
0,108105,2020-08-31,2020-09-18,C,100000,3402.5,3408.2,0,611,133485146,1,0,1,CBOE S&P 500 INDEX,E
1,108105,2020-08-31,2020-09-18,C,1000000,2503.2,2508.0,0,37663,130915017,1,0,1,CBOE S&P 500 INDEX,E
2,108105,2020-08-31,2020-09-18,C,1100000,2403.4,2407.9,0,96,130915018,1,0,1,CBOE S&P 500 INDEX,E
3,108105,2020-08-31,2020-09-18,C,1200000,2303.5,2307.9,0,19,129372797,1,0,1,CBOE S&P 500 INDEX,E
4,108105,2020-08-31,2020-09-18,C,1225000,2278.8,2283.1,0,0,133142190,1,0,1,CBOE S&P 500 INDEX,E


In [215]:
missing = df.isna().sum() # column sum
missing = missing[missing > 0] # filter keeping columns with at least a missing value
pct = (missing / len(df) * 100).round(2) # percentage
pd.DataFrame({"missing_count": missing, "missing_pct": pct}).sort_values("missing_pct", ascending=False)

,missing_count,missing_pct


### 'date'

In [216]:
dates = pd.to_datetime(df["date"], errors="coerce")
print(f"Range date: {dates.min().date()} -> {dates.max().date()}")
print(f"Unique days present: {dates.dt.date.nunique()}")

Range date: 2020-08-31 -> 2025-08-29
Unique days present: 1256


Check if dataset has all trading days that should have in the chosen time period

In [217]:
span_years = (dates.max() - dates.min()).days / 365.25 # leap years
expected_approx = int(span_years * 252) # total trading days
print(f"Expected trading days (estimate): ~{expected_approx}")
print(f"Actual days present: {dates.dt.date.nunique()}")
if dates.dt.date.nunique() < expected_approx * 0.85:
    print("[!] WARNING: coverage significantly below expected")

Expected trading days (estimate): ~1258
Actual days present: 1256


### 'exdate'

In [218]:
exdate = pd.to_datetime(df["exdate"], errors="coerce")
date = pd.to_datetime(df["date"], errors="coerce")
days_to_exp = (exdate - date).dt.days

print(f"Day to expiration: min={days_to_exp.min()}, max={days_to_exp.max()}, media={days_to_exp.mean():.0f}")
n_over_year = (days_to_exp > 365).sum()
print(f"Rows with expiration > 365 days: {n_over_year}")
if n_over_year > 0:
    print("[!] WARNING: the query filter did not exclude everything.")

Day to expiration: min=0, max=356, media=77
Rows with expiration > 365 days: 0


### 'cp_flag'

In [219]:
df["cp_flag"].value_counts()

cp_flag
C    12235448
P    12235445
Name: count, dtype: int64

### 'strike_price'

Visualize range of strikes

In [220]:
print(f"raw strike_price: min={df['strike_price'].min()}, max={df['strike_price'].max()}")
print(f"strike_price /1000: min={df['strike_price'].min()/1000:.2f}, max={df['strike_price'].max()/1000:.2f}")

raw strike_price: min=100000, max=12000000
strike_price /1000: min=100.00, max=12000.00


In [221]:
strikes = df["strike_price"] / 1000
print(f"5%: {strikes.quantile(0.05):.2f} | 50%: {strikes.quantile(0.5):.2f} | 95%: {strikes.quantile(0.95):.2f}")

5%: 2200.00 | 50%: 4275.00 | 95%: 6290.00


### 'best_bid and best_offer'

In [222]:
print(f"negative best_bid: {(df['best_bid'] < 0).sum()}")
print(f"negative best_offer: {(df['best_offer'] < 0).sum()}")
n_crossed = (df["best_offer"] < df["best_bid"]).sum()
print(f"Rows with ask < bid (anomalous): {n_crossed}")

negative best_bid: 0
negative best_offer: 0
Rows with ask < bid (anomalous): 40


### 'volume', 'open_interest'

In [223]:
n_total = len(df)
n_zero_vol = (df["volume"] == 0).sum()
n_zero_oi = (df["open_interest"] == 0).sum()
n_both_zero = ((df["volume"] == 0) & (df["open_interest"] == 0)).sum()
n_remaining = n_total - n_both_zero

print(f"Total rows: {n_total:,}")
print(f"volume=0: {n_zero_vol:,} ({n_zero_vol/n_total*100:.1f}%)")
print(f"open_interest=0: {n_zero_oi:,} ({n_zero_oi/n_total*100:.1f}%)")
print(f"Both zero: {n_both_zero:,} ({n_both_zero/n_total*100:.1f}%)")
print(f"Rows remaining after filter: {n_remaining:,} ({n_remaining/n_total*100:.1f}%)")

Total rows: 24,470,893
volume=0: 15,608,735 (63.8%)
open_interest=0: 6,470,723 (26.4%)
Both zero: 6,019,407 (24.6%)
Rows remaining after filter: 18,451,486 (75.4%)


### 'exercise_style'

In [224]:
df["exercise_style"].value_counts()

exercise_style
E    24470893
Name: count, dtype: int64

### 'ss_flag'

In [225]:
df["ss_flag"].value_counts()

ss_flag
0    24470893
Name: count, dtype: int64

### 'index_flag'

In [226]:
df["index_flag"].value_counts()

index_flag
1    24470893
Name: count, dtype: int64

In [227]:
df["_year"] = pd.to_datetime(df["date"]).dt.year
df[df["volume"] > 0].groupby("_year").size()

_year
2020     415101
2021    1396526
2022    1754753
2023    1818921
2024    1992097
2025    1484760
dtype: int64

---
## 3. Security Prices — AAPL

In [228]:
df = pd.read_csv("data/raw/AAPL_security_prices.csv")
print(f"Righe: {len(df):,}")
print(f"Colonne: {list(df.columns)}")
df.head()

Righe: 1,256
Colonne: ['secid', 'date', 'low', 'high', 'open', 'close', 'volume', 'return', 'cfadj', 'cfret']


,secid,date,low,high,open,close,volume,return,cfadj,cfret
0,101594,2020-08-31,126.00,131.00,127.58,129.04,225702688,0.033912,112,129.6245
1,101594,2020-09-01,130.53,134.80,132.76,134.18,152470142,0.039833,112,129.6245
2,101594,2020-09-02,127.00,137.98,137.59,131.40,200118991,-0.020718,112,129.6245
3,101594,2020-09-03,120.50,128.84,126.91,120.88,257599640,-0.080061,112,129.6245
4,101594,2020-09-04,110.89,123.70,120.07,120.96,332607163,0.000662,112,129.6245


In [229]:
missing = df.isna().sum()
missing = missing[missing > 0]
pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": pct}).sort_values("missing_pct", ascending=False)

,missing_count,missing_pct


In [230]:
dates = pd.to_datetime(df["date"], errors="coerce")
print(f"Range date: {dates.min().date()} -> {dates.max().date()}")
print(f"Unique days present: {dates.dt.date.nunique()}")

Range date: 2020-08-31 -> 2025-08-29
Unique days present: 1256


In [231]:
print(df["cfadj"].value_counts())
print(df["cfadj"].nunique())
print(df["cfadj"].unique())

cfadj
112    1256
Name: count, dtype: int64
1
[112]


---
## 4. Security Prices — SPX

In [232]:
df = pd.read_csv("data/raw/SPX_security_prices.csv")
print(f"Righe: {len(df):,}")
print(f"Colonne: {list(df.columns)}")
df.head()

Righe: 1,256
Colonne: ['secid', 'date', 'low', 'high', 'open', 'close', 'volume', 'return', 'cfadj', 'cfret']


,secid,date,low,high,open,close,volume,return,cfadj,cfret
0,108105,2020-08-31,3493.25,3514.77,3509.73,3500.31,0,-0.002195,1,1
1,108105,2020-09-01,3494.60,3528.03,3507.44,3526.65,0,0.007525,1,1
2,108105,2020-09-02,3535.23,3588.11,3543.76,3580.84,0,0.015366,1,1
3,108105,2020-09-03,3427.41,3564.85,3564.74,3455.06,0,-0.035126,1,1
4,108105,2020-09-04,3349.63,3479.15,3453.60,3426.96,0,-0.008133,1,1


In [233]:
missing = df.isna().sum()
missing = missing[missing > 0]
pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": pct}).sort_values("missing_pct", ascending=False)

,missing_count,missing_pct


In [234]:
dates = pd.to_datetime(df["date"], errors="coerce")
print(f"Range date: {dates.min().date()} -> {dates.max().date()}")
print(f"Giorni unici presenti: {dates.dt.date.nunique()}")

Range date: 2020-08-31 -> 2025-08-29
Giorni unici presenti: 1256


---
## 5. Dividends — AAPL (Dividend Distribution History)

In [235]:
df = pd.read_csv("data/raw/AAPL_dividend_yield.csv")
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head()

Rows: 20
Columns: ['secid', 'record_date', 'ex_date', 'amount', 'adj_factor', 'distr_type', 'frequency', 'currency', 'approx_flag', 'cancel_flag', 'liquid_flag']


,secid,record_date,ex_date,amount,adj_factor,distr_type,frequency,currency,approx_flag,cancel_flag,liquid_flag
0,101594,2020-11-09,2020-11-06,0.205,1,1,3,USD,0,0,0
1,101594,2021-02-08,2021-02-05,0.205,1,1,3,USD,0,0,0
2,101594,2021-05-10,2021-05-07,0.220,1,1,3,USD,0,0,0
3,101594,2021-08-09,2021-08-06,0.220,1,1,3,USD,0,0,0
4,101594,2021-11-08,2021-11-05,0.220,1,1,3,USD,0,0,0


In [236]:
missing = df.isna().sum()
missing = missing[missing > 0]
pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": pct}).sort_values("missing_pct", ascending=False)

,missing_count,missing_pct


In [ ]:
print("Rows with cancel_flag=1 (da escludere):", (df["cancel_flag"] == 1).sum())
print("Rows with liquid_flag=1 (da escludere):", (df["liquid_flag"] == 1).sum())

Righe con cancel_flag=1 (da escludere): 0
Righe con liquid_flag=1 (da escludere): 0


In [163]:
df["distr_type"].value_counts()

distr_type
1    20
Name: count, dtype: int64

In [164]:
print(df["currency"].value_counts())
print(df["frequency"].value_counts())

currency
USD    20
Name: count, dtype: int64
frequency
3    20
Name: count, dtype: int64


---
## 6. Dividend Yield — SPX (Index Dividend Yield)

In [165]:
df = pd.read_csv("data/raw/SPX_dividend_yield.csv")
print(f"Righe: {len(df):,}")
print(f"Colonne: {list(df.columns)}")
df.head()

Righe: 1,256
Colonne: ['secid', 'date', 'rate']


,secid,date,rate
0,108105,2020-08-31,1.588993
1,108105,2020-09-01,1.590263
2,108105,2020-09-02,1.572741
3,108105,2020-09-03,1.577477
4,108105,2020-09-04,1.582210


In [166]:
missing = df.isna().sum()
missing = missing[missing > 0]
pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": pct}).sort_values("missing_pct", ascending=False)

,missing_count,missing_pct


In [167]:
dates = pd.to_datetime(df["date"], errors="coerce")
print(f"Range date: {dates.min().date()} -> {dates.max().date()}")
print(f"Giorni unici presenti: {dates.dt.date.nunique()}")

Range date: 2020-08-31 -> 2025-08-29
Giorni unici presenti: 1256


In [168]:
print(f"rate: min={df['rate'].min():.4f}, max={df['rate'].max():.4f}, media={df['rate'].mean():.4f}")
if df["rate"].max() > 0.20 or df["rate"].min() < -0.01:
    print("[!] ATTENZIONE: valori di rate fuori da un range plausibile.")

rate: min=0.6496, max=1.6807, media=1.2208
[!] ATTENZIONE: valori di rate fuori da un range plausibile.


---
## 7. Historical Volatility — AAPL

In [181]:
df = pd.read_csv("data/raw/AAPL_historical_volatility.csv")
print(f"Righe: {len(df):,}")
print(f"Colonne: {list(df.columns)}")
df.head()

Righe: 16,328
Colonne: ['secid', 'date', 'days', 'volatility', 'index_flag']


,secid,date,days,volatility,index_flag
0,101594,2020-08-31,10,0.267962,0
1,101594,2020-08-31,14,0.304956,0
2,101594,2020-08-31,30,0.313118,0
3,101594,2020-08-31,60,0.375779,0
4,101594,2020-08-31,91,0.352131,0


In [182]:
missing = df.isna().sum()
missing = missing[missing > 0]
pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": pct}).sort_values("missing_pct", ascending=False)

,missing_count,missing_pct


In [183]:
dates = pd.to_datetime(df["date"], errors="coerce")
print(f"Range date: {dates.min().date()} -> {dates.max().date()}")
print(f"Giorni unici presenti: {dates.dt.date.nunique()}")

Range date: 2020-08-31 -> 2025-08-29
Giorni unici presenti: 1256


In [184]:
df["days"].value_counts().sort_index()

days
10      1256
14      1256
30      1256
60      1256
91      1256
122     1256
152     1256
182     1256
273     1256
365     1256
547     1256
730     1256
1825    1256
Name: count, dtype: int64

In [185]:
df["volatility"].describe()

count    16328.000000
mean         0.287230
std          0.091004
min          0.046444
25%          0.218581
50%          0.286139
75%          0.335358
max          1.217727
Name: volatility, dtype: float64

In [186]:
os.getcwd()

'c:\\Users\\glmar\\OneDrive\\Bemacs Bocconi\\TESI\\Thesis\\Priced-by-intelligence'

We verify empirically that the `volatility` field in the OptionMetrics Historical_Volatility file is annualized. Manually recomputing the 60-day realized volatility from raw log-returns for a single reference date (2025-08-29) and annualizing by √252 yields 0.2168, closely matching the reported value of 0.2255. The small residual difference is likely attributable to OptionMetrics' use of 60 calendar days (approximately 42 trading days) rather than 60 trading-day observations. This single-date check confirms the annualization convention; `sigma` is therefore used directly in the Black-Scholes formula without further scaling.

In [187]:
# Load both files fresh, to keep this check self-contained
aapl_hv_check = pd.read_csv("data/raw/AAPL_historical_volatility.csv")
aapl_hv_check["date"] = pd.to_datetime(aapl_hv_check["date"])

aapl_prices_check = pd.read_csv("data/raw/AAPL_security_prices.csv")
aapl_prices_check["date"] = pd.to_datetime(aapl_prices_check["date"])

# Pick a reference date to test (use the most recent 60-day volatility record available)
test_date = aapl_hv_check[aapl_hv_check["days"] == 60]["date"].max()
reported_vol = aapl_hv_check[
    (aapl_hv_check["days"] == 60) & (aapl_hv_check["date"] == test_date)
]["volatility"].values[0]

# Manually recompute: 60 most recent daily log returns strictly before test_date
prices_before = aapl_prices_check[aapl_prices_check["date"] < test_date].sort_values("date").tail(61)
log_returns = np.log(prices_before["close"] / prices_before["close"].shift(1)).dropna()

manual_daily_std = log_returns.std()
manual_annualized_std = manual_daily_std * np.sqrt(252)

print(f"Test date: {test_date}")
print(f"Reported volatility (file): {reported_vol:.4f}")
print(f"Manually computed DAILY std: {manual_daily_std:.4f}")
print(f"Manually computed ANNUALIZED std (daily * sqrt(252)): {manual_annualized_std:.4f}")

Test date: 2025-08-29 00:00:00
Reported volatility (file): 0.2255
Manually computed DAILY std: 0.0137
Manually computed ANNUALIZED std (daily * sqrt(252)): 0.2168


---
## 8. Historical Volatility — SPX

In [213]:
df = pd.read_csv("data/raw/SPX_historical_volatility.csv")
print(f"Righe: {len(df):,}")
print(f"Colonne: {list(df.columns)}")
df.head()

Righe: 16,328
Colonne: ['secid', 'date', 'days', 'volatility', 'index_flag']


,secid,date,days,volatility,index_flag
0,108105,2020-08-31,10,0.077505,1
1,108105,2020-08-31,14,0.074191,1
2,108105,2020-08-31,30,0.080902,1
3,108105,2020-08-31,60,0.111899,1
4,108105,2020-08-31,91,0.194460,1


In [214]:
missing = df.isna().sum()
missing = missing[missing > 0]
pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": pct}).sort_values("missing_pct", ascending=False)

,missing_count,missing_pct


In [215]:
dates = pd.to_datetime(df["date"], errors="coerce")
print(f"Range date: {dates.min().date()} -> {dates.max().date()}")
print(f"Giorni unici presenti: {dates.dt.date.nunique()}")

Range date: 2020-08-31 -> 2025-08-29
Giorni unici presenti: 1256


In [216]:
df["days"].value_counts().sort_index()

days
10      1256
14      1256
30      1256
60      1256
91      1256
122     1256
152     1256
182     1256
273     1256
365     1256
547     1256
730     1256
1825    1256
Name: count, dtype: int64

---
## 9. Zero Coupon Yield Curve 

In [188]:
df = pd.read_csv("data/raw/riskfree_rate.csv")
print(f"Righe: {len(df):,}")
print(f"Colonne: {list(df.columns)}")
df.head()

Righe: 24,764
Colonne: ['date', 'days', 'rate']


,date,days,rate
0,2020-08-31,7,0.109752
1,2020-08-31,17,0.130237
2,2020-08-31,52,0.199518
3,2020-08-31,80,0.214845
4,2020-08-31,108,0.228795


In [189]:
missing = df.isna().sum()
missing = missing[missing > 0]
pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": pct}).sort_values("missing_pct", ascending=False)

,missing_count,missing_pct


In [190]:
dates = pd.to_datetime(df["date"], errors="coerce")
print(f"Range date: {dates.min().date()} -> {dates.max().date()}")
print(f"Giorni unici presenti: {dates.dt.date.nunique()}")

Range date: 2020-08-31 -> 2025-08-29
Giorni unici presenti: 1256


In [191]:
print(f"rate: min={df['rate'].min():.4f}, max={df['rate'].max():.4f}, media={df['rate'].mean():.4f}")
if df["rate"].max() > 0.20 or df["rate"].min() < -0.01:
    print("[!] ATTENZIONE: valori di rate fuori da un range plausibile.")

rate: min=0.0587, max=5.7935, media=2.1507
[!] ATTENZIONE: valori di rate fuori da un range plausibile.


In [192]:
sorted(df["days"].unique())

[np.int64(7),
 np.int64(8),
 np.int64(10),
 np.int64(11),
 np.int64(12),
 np.int64(13),
 np.int64(14),
 np.int64(15),
 np.int64(17),
 np.int64(18),
 np.int64(19),
 np.int64(20),
 np.int64(21),
 np.int64(22),
 np.int64(24),
 np.int64(25),
 np.int64(26),
 np.int64(27),
 np.int64(28),
 np.int64(29),
 np.int64(30),
 np.int64(31),
 np.int64(32),
 np.int64(33),
 np.int64(34),
 np.int64(35),
 np.int64(36),
 np.int64(38),
 np.int64(39),
 np.int64(40),
 np.int64(41),
 np.int64(42),
 np.int64(43),
 np.int64(45),
 np.int64(46),
 np.int64(47),
 np.int64(48),
 np.int64(49),
 np.int64(50),
 np.int64(52),
 np.int64(53),
 np.int64(54),
 np.int64(55),
 np.int64(56),
 np.int64(57),
 np.int64(59),
 np.int64(60),
 np.int64(61),
 np.int64(62),
 np.int64(63),
 np.int64(64),
 np.int64(66),
 np.int64(67),
 np.int64(68),
 np.int64(69),
 np.int64(70),
 np.int64(71),
 np.int64(73),
 np.int64(74),
 np.int64(75),
 np.int64(76),
 np.int64(77),
 np.int64(78),
 np.int64(80),
 np.int64(81),
 np.int64(82),
 np.int64(83

In [193]:
# For every date in the raw riskfree curve, check how far the closest available tenor is from 90 days
riskfree_check = df.copy()
riskfree_check["days_diff"] = (riskfree_check["days"] - 90).abs()
closest_per_date = riskfree_check.groupby("date")["days_diff"].min()

print(closest_per_date.describe())
print()
print("Dates where the closest tenor is more than 5 days away from 90:", (closest_per_date > 5).sum())

count    1256.000000
mean        2.695064
std         3.701858
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        17.000000
Name: days_diff, dtype: float64

Dates where the closest tenor is more than 5 days away from 90: 218
